# 用 Cython 提速

用循环逐个计算数组元素时，Python 往往比 C 慢几百倍，这一章以距离矩阵的计算为例，看 Cython 能把它提到多快。

- 用 `%%cython` 魔法命令当场编译试跑
- 用 `cdef` 声明类型，把循环变成 C 的循环
- 用 `setup.py` 或 `cythonize` 做成扩展模块
- 用内存视图高效地处理 NumPy 数组
- 调用 Python/C API，快速做出列表与元组

## 计算向量的距离矩阵

先写一个函数，把所有组合的向量之间的距离都算出来，数组 `X` 的形状是 `(200, 3)`，可以看成三维空间里的 200 个点。

In [1]:
import numpy as np

np.random.seed(42)
X = np.random.rand(200, 3)

距离用 NumPy 的广播也能算，用 `scipy.spatial.distance.pdist()` 也能算。这里的目的是比较 Python 与 Cython 的速度，所以用三重循环一个一个算。

In [2]:
def pairwise_dist_python(X):
    m, n = X.shape
    D = np.empty((m, m), dtype=np.float64)
    for i in range(m):
        for j in range(i, m):
            d = 0.0
            for k in range(n):
                tmp = X[i, k] - X[j, k]
                d += tmp * tmp
            D[i, j] = D[j, i] = d**0.5
    return D

为了确认结果对不对，也用 `pdist()` 做同样的计算，并比较速度：Python 的循环大约比 `pdist()` 慢 200 倍，这个倍数随机器与当时的负载会有波动。

In [3]:
from scipy.spatial.distance import pdist, squareform

%timeit squareform(pdist(X))
%timeit pairwise_dist_python(X)
np.allclose(squareform(pdist(X)), pairwise_dist_python(X))

207 μs ± 52.1 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


46.7 ms ± 1.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


True

### `%%cython` 魔法命令

Cython 的程序需要编译，通常要写 `setup.py`，那放到后面说明，先用 Notebook 的 `%%cython` 魔法命令当场编译运行。

`%%cython` 写在格子的开头，它把整个格子用 Cython 编译成扩展模块，再载入里面的对象。因为是另一个模块，所以里面也要 `import numpy`。

下面的程序与 `pairwise_dist_python()` 相同，但没有声明类型，所以速度几乎不变。

In [4]:
%load_ext helper.cython

In [5]:
%%cython
import numpy as np

def pairwise_dist_cython(X):
    m, n = X.shape
    D = np.empty((m, m), dtype=np.float64)
    for i in range(m):
        for j in range(i, m):
            d = 0.0
            for k in range(n):
                tmp = X[i, k] - X[j, k]
                d += tmp * tmp
            D[i, j] = D[j, i] = d ** 0.5
    return D

测一下速度。

In [6]:
%timeit pairwise_dist_cython(X)
np.allclose(pairwise_dist_cython(X), pairwise_dist_python(X))

47.3 ms ± 977 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


True

下面的程序用上了 Cython 的全部优化，`cdef` 是声明变量类型的关键字，`@cython.boundscheck(False)` 之类则是编译的指示。

* 参数与返回值写成 `double[:, ::1]`，把 NumPy 数组当作 C 的二维数组来处理。
* 用 `cdef int`、`cdef double` 把循环变量与计算变成 C 的类型。
* 用 `from libc.math cimport sqrt` 调用 C 的 `sqrt()`。

这样一来，循环内部就是完全不碰 Python 对象的 C 代码。

In [7]:
%%cython
import numpy as np
import cython
from libc.math cimport sqrt

@cython.boundscheck(False)
@cython.wraparound(False)
def pairwise_dist_cython2(double[:, ::1] X):
    cdef int m, n, i, j, k
    cdef double tmp, d
    m, n = X.shape[0], X.shape[1]    
    cdef double[:, ::1] D = np.empty((m, m), dtype=np.float64)
    for i in range(m):
        for j in range(i, m):
            d = 0.0
            for k in range(n):
                tmp = X[i, k] - X[j, k]
                d += tmp * tmp
            D[i, j] = D[j, i] = sqrt(d)
    return np.asarray(D)

带类型的版本比 `pdist()` 大约快 2 倍。

In [8]:
%timeit pairwise_dist_cython2(X)
np.allclose(pairwise_dist_cython2(X), pairwise_dist_python(X))

111 μs ± 874 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


True

### 用 Cython 做扩展模块

在 Notebook 里确认了速度与正确性之后，就把它做成别的程序也能调用的扩展模块。`%%cython` 内部也是在生成扩展模块，所以可以从 `sys.modules` 里找到那个文件。

In [9]:
import sys

sys.modules[pairwise_dist_cython2.__module__]

<module '_cython_magic_7d3efdcf7eb9c3c3e4be9347e85f3569e5f70f81569488022595792ad9ce53b6' from 'C:\\Users\\ruoyu\\.ipython\\cython\\_cython_magic_7d3efdcf7eb9c3c3e4be9347e85f3569e5f70f81569488022595792ad9ce53b6.cp314-win_amd64.pyd'>

要把 Cython 的程序做成扩展模块，就写一个 `setup_fast_pdist.py` 这样的安装脚本：用 `Extension` 定义名为 `fast_pdist` 的模块，再指定 Cython 的源码 `fast_pdist.pyx`；因为要用 NumPy，所以用 `np.get_include()` 把头文件的位置传过去。

In [10]:
%%writefile setup_fast_pdist.py
from distutils.core import setup
from distutils.extension import Extension
from Cython.Distutils import build_ext
import numpy as np

ext_modules = [
    Extension("fast_pdist", ["fast_pdist.pyx"],
        include_dirs = [np.get_include()]),    
]

setup(
  name = 'a faster version of pdist',
  cmdclass = {'build_ext': build_ext},
  ext_modules = ext_modules
)

Writing setup_fast_pdist.py


In [11]:
%%writefile fast_pdist.pyx
# cython: language_level=3
import numpy as np
import cython
from libc.math cimport sqrt

@cython.boundscheck(False)
@cython.wraparound(False)
def pairwise_dist_cython2(double[:, ::1] X):
    cdef int m, n, i, j, k
    cdef double tmp, d
    m, n = X.shape[0], X.shape[1]    
    cdef double[:, ::1] D = np.empty((m, m), dtype=np.float64)
    for i in range(m):
        for j in range(i, m):
            d = 0.0
            for k in range(n):
                tmp = X[i, k] - X[j, k]
                d += tmp * tmp
            D[i, j] = D[j, i] = sqrt(d)
    return np.asarray(D)

Overwriting fast_pdist.pyx


在命令行执行下面的命令，`fast_pdist.pyx` 就会被编译出 `fast_pdist.pyd`：加上 `--inplace` 之后，扩展模块与源码放在同一个位置。

In [12]:
!{sys.executable} setup_fast_pdist.py build_ext --inplace

Compiling fast_pdist.pyx because it changed.
[1/1] Cythonizing fast_pdist.pyx


载入做好的扩展模块，调用函数。

In [13]:
import fast_pdist

np.allclose(fast_pdist.pairwise_dist_cython2(X), pairwise_dist_python(X))

True

不写 `setup.py` 也能用 `cythonize -i` 编译，`-i` 的意思与 `--inplace` 相同。

:::{warning}
重新编译已经载入过的扩展模块时，会因为覆盖不了 `.pyd` 而出错，原因是 Windows 上那个文件正在被使用，这时可以重启内核，或者像下面这样换个名字编译。
:::

In [14]:
import shutil
from Cython.Build.Cythonize import main as cythonize_main

shutil.copy("fast_pdist.pyx", "fast_pdist_build.pyx")  # 已载入的 .pyd 覆盖不了
sys.argv = ["cythonize", "-i", "fast_pdist_build.pyx"]
cythonize_main()

Compiling C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\fast_pdist_build.pyx because it changed.
[1/1] Cythonizing C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\fast_pdist_build.pyx


## Cython 的基本语法

这一节一边看 Cython 生成的 C 代码，一边说明语法。

### 用 C 表示 Python 的对象

Cython 的程序先由 Cython 编译器转换成 C，再由 C 编译器做成扩展模块。Cython 里既能处理 C 的类型，也能处理 Python 的对象，所以先看 C 这一层是怎么表示 Python 对象的。

Python 本身是用 C 写的，任何对象都是 C 的结构体，而开头两个字段在任何类型里含义都相同。

* `ob_refcnt`：引用计数，减到 0 时那块内存被释放。
* `ob_type`：指向类型对象的指针。

C 的程序里定义了 `PyObject` 结构体，它只有这两个字段；别的类型则在它后面追加字段，例如 `float` 的结构体如下。

```c
typedef struct {
    Py_ssize_t ob_refcnt;
    struct _typeobject *ob_type;
    double ob_fval;
} PyFloatObject;
```

In [15]:
import sys

sys.getsizeof(1.0)

24

任何对象的开头都一样，所以 C 里用 `PyObject *` 表示 Python 的对象。Cython 里也一样：不声明类型的变量就是 Python 的对象，转成 C 之后就是 `PyObject *`。

Python 提供了写 C 扩展模块用的 Python/C API。给 `%%cython` 加上 `-a` 就能看到 Cython 的代码与编译后的 C 代码的对应关系，颜色越深的行执行起来可能越慢；C 的代码由 Cython 自动生成，不必细读。

In [16]:
%%cython -a
a = 1.0
b = 2.0
c = a + b

上面程序里的 `a`、`b`、`c` 全是 Python 的对象，所以 `c = a + b` 会变成下面这样的 C 代码，其中 `__Pyx_GOTREF` 与 `__Pyx_DECREF` 负责增减引用计数。

```c
  __Pyx_GetModuleGlobalName(__pyx_t_2, __pyx_n_s_a); if (unlikely(!__pyx_t_2)) __PYX_ERR(0, 3, __pyx_L1_error)
  __Pyx_GOTREF(__pyx_t_2);
  __Pyx_GetModuleGlobalName(__pyx_t_3, __pyx_n_s_b); if (unlikely(!__pyx_t_3)) __PYX_ERR(0, 3, __pyx_L1_error)
  __Pyx_GOTREF(__pyx_t_3);
  __pyx_t_4 = PyNumber_Add(__pyx_t_2, __pyx_t_3); if (unlikely(!__pyx_t_4)) __PYX_ERR(0, 3, __pyx_L1_error)
  __Pyx_GOTREF(__pyx_t_4);
  __Pyx_DECREF(__pyx_t_2); __pyx_t_2 = 0;
  __Pyx_DECREF(__pyx_t_3); __pyx_t_3 = 0;
  if (PyDict_SetItem(__pyx_d, __pyx_n_s_c, __pyx_t_4) < 0) __PYX_ERR(0, 3, __pyx_L1_error)
  __Pyx_DECREF(__pyx_t_4); __pyx_t_4 = 0;
```

### 用 `cdef` 声明变量的类型

要想跑得快，就用 `cdef` 给常用的变量加上类型，效果有两个。

* 变量的读写变快：Python 用字典保存名字与值的对应关系，每次读写都要查字典；用 `cdef` 声明之后，对应关系在编译时就定下来，不必再查。
* 计算变快：Python 连数值计算也要调用 API，而类型已知时 Cython 可以直接用 C 的运算。

下面的例子把 `a`、`b`、`c` 声明成 `double`，输出的 C 如下。

```c
static double a, b, c;
...
PyMODINIT_FUNC init(){
    ...
    a = 1.0;
    b = 2.0;
    c = a + b;
    ...
}
```

:::{warning}
用 `cdef` 声明的全局变量是 C 的变量，所以从 Python 里读不到它的值。
:::

In [17]:
%%cython -a
cdef double a = 1.0
cdef double b = 2.0
cdef double c = a + b

把 C 的变量与 Python 的对象一起计算时，Cython 先把它转换成 Python 的对象，再调用 API。下面的例子里 `s` 是 C 的变量，`a` 是 Python 的 `float`。

❶ 直接把 `s` 与 `a` 相加，会执行下面这段 C 代码，其中 `_v_s` 是 `double`，`_t_1` 到 `_t_3` 是 Python 的对象，`_s_a` 是字符串 `"a"`。

```c
_t_1 = PyFloat_FromDouble(_v_s); //把 s 转换成 Python 的对象
__Pyx_GetModuleGlobalName(_t_2, _s_a); //从全局字典取出 "a" 对应的对象
_t_3 = PyNumber_Add(_t_1, _t_2); //加法运算
_v_s = __pyx_PyFloat_AsDouble(_t_3); //把加法运算的结果转换成 double
```

❷ 用 `<double>` 把 Python 的对象转换成 `double`，就成了 C 的加法。

```c
__Pyx_GetModuleGlobalName(_t_1, _s_a); 
_t_2 = __pyx_PyFloat_AsDouble(_t_1); 
_v_s = _v_s + _t_2;
```

In [18]:
%%cython -a
cdef double s = 0
a = 3.0
s = s + a #❶
s = s + <double>a #❷

除了 C 的类型，Cython 也认识 Python 的内置类型(列表、字典、元组等)。下面的例子把 `clist` 声明成列表、`cindex` 声明成 `int`，再取出元素；为了比较，用不声明类型的 `pylist` 与 `pyindex` 做同样的事。

In [19]:
%%cython -a
cdef list clist = [1000, 2, 3]
cdef int cindex = 0
clist[cindex] #❶

pylist = [1000, 2, 3]
pyindex = 0
pylist[pyindex] #❷

❶ `clist` 与 `cindex` 的类型已知，所以 `clist[cindex]` 变成调用取列表元素的函数 `__Pyx_GetItemInt_List()`。

```c
__Pyx_GetItemInt_List(clist, cindex, int, 1, __Pyx_PyInt_From_int, 1, 1, 1)

```

❷ 没有声明类型的 `pylist` 与 `pyindex` 被当作 Python 的对象，`pylist[pyindex]` 变成下面这段代码。

```c
__Pyx_GetModuleGlobalName(__pyx_t_1, __pyx_n_s_pylist);
__Pyx_GetModuleGlobalName(__pyx_t_2, __pyx_n_s_pyindex);
__pyx_t_3 = __Pyx_PyObject_GetItem(__pyx_t_1, __pyx_t_2);
```

先用 `__Pyx_GetModuleGlobalName()` 从全局字典取出两个对象，再用 `__Pyx_PyObject_GetItem()` 取元素。

不只是 `[]`，Cython 还认识一些方法与属性，下表中的写法都会变成高效的代码。

| 数据类型  | 变快的运算  |
|:--|:--|
| `bytes`、`str`  | `join()`、`in`  |
| `tuple`  | `in`  |
| `list`  | `in`、`insert()`、`reverse()`、`append()`、`extend()`  |
| `dict`  | `in`、`get()`、`keys()`、`values()`、`clear()`、`copy()` |
| `set`   | `in`、`clear()`、`add()`、`pop()` |
| `slice` | `start`、`stop`、`step` |
| `complex` | `real`、`imag` |

给前面的程序加上 `clist.append(pyindex)`，就能看到 `append()` 被编译成什么；Cython 不认识的属性与方法，都会变成查找 Python 属性的 API 调用。

### 用 `def` 定义函数

通常用 `def` 定义函数、从 Python 调用。既然是从 Python 调用，参数与返回值就是 Python 的对象；不过 Cython 里可以给参数加类型，由 Cython 检查类型并转换成 C 的类型。

下面 `py_square_add()` 的参数是 `double`：从 Python 传 `float` 进去，里面转换成 `double` 计算，再把结果变回 `float`。C 的代码里 `x`、`y`、`_r` 是 Python 的对象，`_v_x`、`_v_y` 是 `double`，转换用 `PyFloat_AsDouble()` 与 `PyFloat_FromDouble()`。

In [20]:
%%cython -a
def py_square_add(double x, double y):
    return x*x + y*y

把参数声明成 `list` 这样的 Python 类型时，Cython 会检查类型，并把认识的运算变成更快的代码：下面的例子里 `alist` 是 `list`，所以 `len(alist)` 变成 `PyList_GET_SIZE()`、`alist[i]` 变成 `__Pyx_GetItemInt_List()`。

In [21]:
%%cython -a
def sum_list(list alist): 
    cdef double s = 0
    cdef int i = 0
    for i in range(len(alist)):
        s += <double>alist[i]
    return s

### 用 `cdef` 定义 C 的函数

用 `def` 定义的函数走 Python 的调用，所以即使在 Cython 内部调用，也要转换参数与返回值；如果在循环里反复调用，这笔费用就很大。

用 `cdef` 定义的函数只能从 Cython 内部调用，但调用费用与 C 的函数相同。下面 `c_square_add()` 接受 `double` 返回 `double`，而 `a = c_square_add(1.0, 2.0)` 会变成下面的 C。

```c
__pyx_v_a = __pyx_f_c_square_add(1.0, 2.0);
```

:::{tip}
`cdef` 函数如果不写返回值的类型，返回值就变成 Python 的对象。
:::

In [22]:
%%cython -a
cdef double c_square_add(double x, double y):
    return x*x + y*y

cdef double a = c_square_add(1.0, 2.0)

既想从 Cython 内部快速调用，又想从 Python 调用时，就用 `cpdef`：它会同时生成 C 的函数与给 Python 用的包装函数，Cython 内部调用比 `cdef` 稍慢，但比 `def` 快得多。

In [23]:
%%cython -a
cpdef double cp_square_add(double x, double y):
    return x*x + y*y

cp_square_add(1.0, 2.0)

## 高效地处理数组

科学计算里会大量出现数组操作，上一节已经看到内存视图能快速访问数组元素，这一节详细说明它的用法。

### Cython 的内存视图

内存视图像下面这样声明。

```text
cdef 元素的类型[维度的声明] 变量名
```

维度的声明里，`:` 表示一个轴，`::1` 表示那个轴的元素是连续的。

* `[:]`：一维数组
* `[::1]`：一维且连续的数组
* `[:, :]`：二维数组
* `[:, ::1]`：二维，且第 1 轴的元素连续

内存视图与 NumPy 数组一样有 `shape`、`strides`、`base`，但本身不持有数据，数据是从别的 Python 对象或 C 的数组借来的。

在 C 这一层它是个结构体，而当作 Python 对象使用时，Cython 会把它转换成 `MemoryView`。下面的程序里，❶ `buf` 是 C 的二维数组，❷ `view` 是使用这块内存的内存视图。

In [24]:
%%cython
import numpy as np

cdef double buf[3][4]            #❶
cdef double[:, ::1] view = buf   #❷

def fill_value(double value):    
    np.asarray(view).fill(value) #❸

def sum_view():
    return np.sum(view)          #❹    
    
def get_view():
    return view                  #❺

def square_view():               #❻
    cdef int i, j
    for i in range(view.shape[0]):
        for j in range(view.shape[1]):
            view[i, j] *= view[i, j]

用 `get_view()` 取到 `MemoryView`，看它的属性，与 NumPy 数组相同。

In [25]:
import helper.magics

view = get_view()
%C view.shape; view.strides; view.itemsize; view.nbytes

view.shape  view.strides  view.itemsize  view.nbytes
----------  ------------  -------------  -----------
(3, 4)      (32, 8)       8              96         


用 `fill_value()`、`square_view()`、`sum_view()` 操作 C 的数组：`MemoryView` 也能用下标读写元素。

In [26]:
fill_value(3.0)
print(sum_view())
square_view()
print(sum_view())
view[1, 2] = 10
print(sum_view())

36.0
108.0
109.0


把 `MemoryView` 变成 NumPy 数组时，两者共享内存。

In [27]:
arr = np.asarray(view)
arr[1, 0] = 11
print(sum_view())
print(arr)

111.0
[[ 9.  9.  9.  9.]
 [11.  9. 10.  9.]
 [ 9.  9.  9.  9.]]


用 `base` 可以取到持有数据的对象。数据放在 C 的全局数组里，所以 Cython 会做一个表示那个数组的对象，再从它的 `memview` 属性做出 `MemoryView`。

In [28]:
print(view.base.__class__)
print(view.base.memview)

<class '_cython_magic_ea1cd99735eefdd16bac56597e22898d324874159081a9b6d50a2df56f1d31cb.array'>
<MemoryView of 'array' object>


### NumPy 数组与内存视图

这是用内存视图处理 NumPy 数组的例子。

❶ 用 `double[:]` 把 `x` 变成一维的内存视图，因为没有指定连续，所以 `x[i]` 会算成 `x.data + i * x.strides[0]`，其中 `x.data` 是开头的地址、`x.strides[0]` 是元素之间相隔的字节数。

❷ `res` 指定了连续，所以 `res[i]` 就是 `((double *) res.data) + i`，少了一次乘法，因此更快。

❸ 用 `res.base` 取回 NumPy 数组：`base` 不是结构体的字段，所以 Cython 先转换成 `MemoryView`，再用 Python 的 `getattr()` 取。

在循环里处理数组时，可以关掉 `boundscheck` 与 `wraparound`，这样就没有下标范围的检查，C 的代码也更短。

In [29]:
%%cython
import numpy as np
import cython

@cython.boundscheck(False)
@cython.wraparound(False)
def square(double[:] x): #❶
    cdef int i
    cdef double[::1] res = np.empty_like(x)
    for i in range(x.shape[0]):
        res[i] = x[i] * x[i] #❷
    return res.base #❸

内存视图也能像 NumPy 数组那样切片，切片会做出共享原内存的新视图，所以有一点费用，但比 NumPy 的切片快得多。

下面的例子用 `cpdef` 定义 `norm()`，把一维向量当场归一化；`norm_axis()` 按二维数组的轴归一化，`inplace` 为 `True` 时当场改写，为 `False` 时返回新数组。

❶ 内存视图可以给切片赋值，那时会做一个临时的视图，并把数据复制过去，❷ 用切片读内存视图会做出新视图，它会被传给 `norm()`。

:::{tip}
这个例子里用 `#cython:` 的注释设置了 `boundscheck` 与 `wraparound`。
:::

In [30]:
%%cython
#cython: boundscheck=False, wraparound=False
import numpy as np
import cython

cpdef norm(double[:] x):
    cdef double s
    cdef int i
    s = 0
    for i in range(x.shape[0]):
        s += x[i]*x[i]
    s = 1 / s**0.5
    for i in range(x.shape[0]):
        x[i] *= s 

def norm_axis(double[:, :] x, int axis=0, bint inplace=True):
    cdef int i
    cdef double[:, :] data
    if not inplace:
        data = np.empty_like(x)
        data[:] = x #❶
    else:
        data = x
        
    if axis == 1:
        for i in range(data.shape[0]):
            norm(data[i, :]) #❷
    elif axis == 0:
        for i in range(data.shape[1]):
            norm(data[:, i]) #❷ 
            
    return data.base

用取地址运算符 `&` 可以取到内存视图的数据地址，并作为指针传给 C 的函数。

❶ 用 `cimport` 从 C 标准库的 `string.h` 读入 `memcpy()`，它把 `src` 的 `n` 个字节复制到 `dst`。

```c
void *memcpy(void *dst, const void *src, size_t n);
```

❷ 用 `&dst[0]` 取到开头的地址并变成 `double *`；`memcpy()` 接受的是字节数，所以要把 `sizeof(double)` 乘上元素个数。

这里的内存视图是连续的，所以能用 `memcpy()` 一次复制完；不连续时就得借助 `strides` 一个个复制。

In [31]:
%%cython
from libc.string cimport memcpy #❶

def copy_memview(double[::1] src, double[::1] dst):
    memcpy(&dst[0], &src[0], sizeof(double)*dst.shape[0]) #❷

In [32]:
a = np.random.rand(10)
b = np.zeros_like(a)
copy_memview(a, b)
assert np.all(a == b)

用类型转换也能把 C 的指针变成内存视图：如果 `addr` 是 `double *`，那么 `<double[:10]>addr` 就是长度 10 的内存视图，长度也可以是变量。不过内存的申请与释放要当心，否则就成了非法指针，程序会崩掉。

## Python 的对象与 API

Cython 认识 Python 内置类型的运算，所以能生成更快的代码；进一步调用 Python/C API，还能做到只有 C 才做得的事。

### 操作 `list`

Python 里要么往空列表用 `append()` 添加，要么用 `[None] * n` 做好再用下标填，Cython 里调用做列表的 API 会更快。

下面的代码比较用 API 做与用 `append()` 做的速度，`my_range()` 调用了 3 个 API。

* `PyList_New(len)`：做出长度为 `len` 的列表，内容是 NULL(空地址)，这时还不能给 Python 用。
* `PyList_SET_ITEM(list, i, o)`：把第 `i` 个元素设成 `o`，它是 C 的宏，能快速填进 NULL 的位置。
* `Py_INCREF(o)`：增加 `o` 的引用计数，算上列表持有它的那一份。

In [33]:
%%cython --compile-args=-w
#cython: boundscheck=False, wraparound=False
from cpython.list cimport PyList_New, PyList_SET_ITEM #❶
from cpython.ref cimport Py_INCREF

def my_range(int n):
    cdef int i
    cdef object obj #❷
    cdef list result
    result = PyList_New(n)
    for i in range(n):
        obj = i
        PyList_SET_ITEM(result, i, obj)
        Py_INCREF(obj)
    return result

def my_range2(int n):
    cdef int i
    cdef list result
    result = []
    for i in range(n):
        result.append(i)
    return result

❶ 用 `cimport` 从 Cython 的头文件里读入 `PyList_New` 与 `PyList_SET_ITEM`：Cython 的头文件以 `.pxd` 结尾，函数与类型的声明方式与 C 的头文件相同。

Cython 安装目录下的 `Includes` 里集中放着这些头文件：`numpy` 里是 NumPy 的类型，`libc` 里是 C 标准库，`cpython` 里是 Python/C API 的声明。

❷ `obj` 是把 C 的 `int` 转换成 Python 的整数后临时持有的变量。

`my_range()` 一开始就做出目标大小的列表，所以没有扩容的费用，比用 `append()` 做大约快 2 倍。

In [34]:
%timeit list(range(100))
%timeit my_range(100)
%timeit my_range2(100)

877 ns ± 27.2 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


1.16 μs ± 134 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


1.62 μs ± 47.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


### 做 `tuple`

`tuple` 不可修改，但在 Cython 里调用 API 就能把内容填进去、快速做出来。下面 `to_tuple_list()` 把二维数组变成元组的列表。

`PyTuple_New` 与 `PyTuple_SET_ITEM` 的用法与列表相同。

In [35]:
%%cython --compile-args=-w
#cython: boundscheck=False, wraparound=False
from cpython.list cimport PyList_New, PyList_SET_ITEM
from cpython.tuple cimport PyTuple_New, PyTuple_SET_ITEM
from cpython.ref cimport Py_INCREF

def to_tuple_list(double[:, :] arr):
    cdef int m, n
    cdef int i, j
    cdef list result
    cdef tuple t
    cdef object obj
    
    m, n = arr.shape[0], arr.shape[1]
    result = PyList_New(m)
    for i in range(m):
        t = PyTuple_New(n)
        for j in range(n):
            obj = arr[i, j]
            PyTuple_SET_ITEM(t, j, obj)
            Py_INCREF(obj)
        PyList_SET_ITEM(result, i, t)
        Py_INCREF(t)
    return result

比较 `tolist()` 与 `to_tuple_list()` 的速度。

In [36]:
import numpy as np

arr = np.random.randint(0, 10, (5, 2)).astype(np.double)
print(to_tuple_list(arr))

arr = np.random.rand(100, 5)
%timeit to_tuple_list(arr)
%timeit arr.tolist()

[(8.0, 9.0), (8.0, 8.0), (5.0, 7.0), (0.0, 9.0), (3.0, 0.0)]


15.6 μs ± 104 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


18.3 μs ± 246 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


### 把 `array.array` 当作动态数组

`array` 模块的 `array` 可以当作一维的动态数组，Cython 里也能用，但调用 `append()` 会走 Python 的函数，所以不会变快；`cpython/array.pxd` 里则有快速操作 `array` 的 `cdef` 函数。

下面 `in_circle()` 从二维的点数组 `points` 里，挑出落在以 `(cx, cy)` 为中心、半径为 `r` 的圆内的点。因为不知道圆内有多少点，所以一个个加到 `array.array` 里。

In [37]:
%%cython -c-Ofast --compile-args=-w
#cython: boundscheck=False, wraparound=False
import numpy as np
from cpython cimport array

def in_circle(double[:, :] points, double cx, double cy, double r):
    cdef array.array[double] res = array.array("d") #❶
    cdef double r2 = r * r
    cdef double p[2] #❷
    cdef int i 
    for i in range(points.shape[0]):
        p[0] = points[i, 0]
        p[1] = points[i, 1]
        if (p[0] - cx)**2 + (p[1] - cy)**2 < r2:
            array.extend_buffer(res, <char*>p, 2) #❸
    return np.frombuffer(res, np.double).copy().reshape(-1, 2) #❹

❶ 用 `cimport` 读入 `array` 的头文件，用 `array.array` 类型声明 `res` 并做出新的 `array.array`，其中 `"d"` 表示双精度浮点数。❷ `p` 是只有 2 个元素的 C 数组，用来临时放正在处理的点。❸ 用 `array.extend_buffer()` 把 `p` 加到 `res` 里，第 1 个参数是数组、第 2 个参数是数据开头的 `char *`、第 3 个参数是元素的个数(不是字节数)。❹ 用 `np.frombuffer()` 做出与 `res` 共享内存的数组，再用 `copy()` 复制一份；不复制的话，`res` 被回收之后数据就没了。

再与 NumPy 的写法比较速度：圆外的点越多，`in_circle()` 就越快。

In [38]:
points = np.random.rand(10000, 2)
cx, cy, r = 0.3, 0.5, 0.05

%timeit points[(points[:, 0] - cx)**2 + (points[:, 1] - cy)**2 < r**2, :]
%timeit in_circle(points, cx, cy, r)

93.5 μs ± 4.98 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


24.9 μs ± 671 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 小结

* 用 `%%cython` 能当场编译试速度；不声明类型就不会变快。
* 用 `cdef` 给变量与函数加上类型，就变成 C 的代码；要从 Python 调用就用 `cpdef`，要接受 Python 的参数就在 `def` 里写类型。
* 数组用 `double[:, ::1]` 这样的内存视图接收，指定连续之后下标更快。
* 循环里加上 `boundscheck=False`、`wraparound=False`，下标的检查就没了。
* 用 `cimport` 读入 C 标准库(`libc`)、Python/C API(`cpython`)、NumPy(`numpy`)的声明。
* 列表与元组用 `PyList_New` / `PyTuple_New` 做更快，但别忘了引用计数的操作。
* 扩展模块用 `setup.py` 或 `cythonize -i` 做；已经载入的 `.pyd` 覆盖不了。